# Postprocessing — Step Search, Best-Step Selection, FAD/F1/Latency

This notebook drives the **postprocessing** half of the Israeli pipeline.
It runs **after** training has produced one or more checkpoints and
**before** `build_deliverables.py` is run locally.

It is the single source of truth for:

1. Batch inference across candidate checkpoints (step search).
2. In-Colab metrics: FAD (VGGish) + latency (GPU).
3. Composite-z step selection.
4. Handoff back to **local Windows** for the F1 metric.

See [`ENGINEERING_DECISIONS.md`](../ENGINEERING_DECISIONS.md) §34–§35 for
the rationale behind the file-naming convention, the `_index.csv` schema,
and the composite z-score formula.

Documentation convention used throughout (see §37): every code cell is
preceded by a markdown header with **What this does / Inputs / Outputs /
Action required / Runtime**.

## Cell 1 — Mount Google Drive and clone the repo

**What this does.** Mounts your personal Drive at `/content/drive` and
clones (or `git pull`s) the project repo so this notebook can import
`postprocessing.run_inference_batch` and friends.

**Inputs.** Your Google Drive account; the `GITHUB_TOKEN` Colab secret
(repo `galgeva1/MusicProject`, branch `main` — already pinned in the cell).

**Outputs.** `/content/MusicProject` checked out; `sys.path` extended.

**Action required.** Approve the OAuth prompt when it appears, and make sure
the `GITHUB_TOKEN` secret is set in the Colab Secrets panel.

**Runtime.** ~30 s.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, pathlib
from google.colab import userdata

_token = userdata.get('GITHUB_TOKEN')
assert _token, 'GITHUB_TOKEN secret is empty — check Colab Secrets panel'
REPO_URL = f'https://{_token}@github.com/galgeva1/MusicProject.git'
BRANCH   = 'main'
REPO_DIR = pathlib.Path('/content/MusicProject')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

# The repo root IS the project root (train.py / paths.py / postprocessing/ live here).
PROJECT_ROOT = REPO_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/MusicProject


## Cell 2 — Install Colab dependencies

**What this does.** Installs the Colab subset of the project requirements.
Use `requirements_colab.txt`; do **not** use `requirements_ml.txt` here —
it pins CPU PyTorch wheels.

**Inputs.** `MusicProject/requirements_colab.txt`.

**Outputs.** Installed Python packages in the Colab runtime.

**Action required.** Restart the runtime if pip prompts you.

**Runtime.** ~2–3 min on a fresh runtime.

In [2]:
!pip install -q -r {PROJECT_ROOT}/requirements_colab.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 132.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!cd {PROJECT_ROOT} && python -m postprocessing.run_inference_batch \
    --run-spec {PROJECT_ROOT}/configs/demo/run_spec_demo_SMOKE.yaml

Device: cuda

=== step 224000  (step_224000.pt) ===
  • Rihanna_Diamonds__step_224000__style_Israeli_Artists_ddim100__role_transferred
  Loaded MIDI: Diamonds.mid -> Piano roll shape: (2, 128, 24375) (onset+sustain)
  Loading BigVGAN model: nvidia/bigvgan_v2_22khz_80band_fmax8k_256x
  (First run will download ~450MB from HuggingFace Hub)
config.json: 100% 1.41k/1.41k [00:00<00:00, 4.98MB/s]
bigvgan_generator.pt: 100% 449M/449M [00:05<00:00, 87.3MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Removing weight norm...
  ✓ BigVGAN [bigvgan_22k] loaded in 8.5s
    Device: cuda
    Config: sr=22050, hop=256, mels=80, fmax=8000

=== step 238000  (step_238000.pt) ===
  • Rihanna_Diamonds__step_238000__style_Israeli_Artists_ddim100__role_transferred
  Loaded MIDI: Diamonds.mid -> Piano roll shape: (2, 128, 243

In [4]:
import json, pathlib
_run = pathlib.Path('/content/drive/MyDrive/MusicProject/versions/Israeli_3style/demo_external/_renders/demo_SMOKE_Diamonds')
info = json.loads((_run / 'timing_infos.json').read_text())
for name, ti in info.items():
    print(f"{ti['song']} step{ti['step']}: infer {ti['infer_s']:.1f}s for {ti['audio_s']:.1f}s audio -> RTF {ti['rtf']:.3f}")

Rihanna_Diamonds step224000: infer 98.9s for 286.9s audio -> RTF 0.345
Rihanna_Diamonds step238000: infer 95.4s for 286.9s audio -> RTF 0.333


In [3]:
import subprocess, pathlib

SPECS = [
    # ddim100 block (~54 min, 35 renders)
    "run_spec_demo_Artists_ddim100.yaml",
    "run_spec_demo_Military_ddim100.yaml",
    "run_spec_demo_Slakh_ddim100.yaml",
    # ddim200 block (~108 min, 35 renders)
    "run_spec_demo_Artists_ddim200.yaml",
    "run_spec_demo_Military_ddim200.yaml",
    "run_spec_demo_Slakh_ddim200.yaml",
]

for name in SPECS:
    spec = pathlib.Path(PROJECT_ROOT) / "configs" / "demo" / name
    print("\n" + "=" * 70)
    print(f"RUNNING: {name}")
    print("=" * 70, flush=True)
    r = subprocess.run(
        ["python", "-m", "postprocessing.run_inference_batch", "--run-spec", str(spec)],
        cwd=str(PROJECT_ROOT),
    )
    print(f"[{name}] exit code = {r.returncode}", flush=True)

print("\nALL SPECS DONE.")


RUNNING: run_spec_demo_Artists_ddim100.yaml
[run_spec_demo_Artists_ddim100.yaml] exit code = 0

RUNNING: run_spec_demo_Military_ddim100.yaml
[run_spec_demo_Military_ddim100.yaml] exit code = 0

RUNNING: run_spec_demo_Slakh_ddim100.yaml
[run_spec_demo_Slakh_ddim100.yaml] exit code = 0

RUNNING: run_spec_demo_Artists_ddim200.yaml
[run_spec_demo_Artists_ddim200.yaml] exit code = 0

RUNNING: run_spec_demo_Military_ddim200.yaml
[run_spec_demo_Military_ddim200.yaml] exit code = 0

RUNNING: run_spec_demo_Slakh_ddim200.yaml
[run_spec_demo_Slakh_ddim200.yaml] exit code = 0

ALL SPECS DONE.


## Cell 3 — Configure paths and pick the version + run id

**What this does.** Defines the absolute Drive paths used by every later
cell: the version directory, the checkpoint directory, and the
`run_spec.yaml` for this step-search.

**Inputs.** `VERSION_NAME` (e.g. `Israeli_Shalom_Arik`), `RUN_ID` (e.g.
`Israeli_Shalom_Arik_step_search_2026_03_01`), and the path to a `run_spec.yaml`
describing which songs, styles, roles, and checkpoint steps to evaluate.

**Outputs.** Python variables — no files written yet.

**Action required.** Edit the three constants below.

**Runtime.** Instant.

In [ ]:
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/MusicProject')

# The 3-version model is ONE checkpoint set (Israeli_3style); we step-search it
# once per STYLE by conditioning on a different version_id (set in the run_spec).
MODEL_VERSION_NAME = 'Israeli_3style'        # where checkpoints + inference_runs live

# --- pick which style's step-search to run -----------------------------------
STYLE = 'Israeli_Artists'                    # or 'Israeli_Military'
RUN_SPEC_PATH = PROJECT_ROOT / 'configs' / f'run_spec_{STYLE}.yaml'

# Derive everything else from the spec so there is a single source of truth.
with open(RUN_SPEC_PATH, 'r') as _f:
    _spec = yaml.safe_load(_f)
RUN_ID   = _spec['run_id']
REAL_DIR = pathlib.Path(_spec['metrics']['fad']['real_dir'])

VERSION_DIR    = DRIVE_ROOT / 'versions' / MODEL_VERSION_NAME
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / MODEL_VERSION_NAME
RUNS_ROOT      = VERSION_DIR / 'inference_runs'

for p in (VERSION_DIR, CHECKPOINT_DIR):
    assert p.exists(), f'missing: {p}'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print('STYLE         =', STYLE)
print('MODEL/CKPTS   =', CHECKPOINT_DIR)
print('RUN_SPEC_PATH =', RUN_SPEC_PATH)
print('RUN_ID        =', RUN_ID)
print('REAL_DIR(FAD) =', REAL_DIR)
print('steps         =', _spec['steps'])


## Cell 4 — Batch inference across candidate checkpoints

**What this does.** Runs `postprocessing.run_inference_batch`, which loops
**steps outer × (song × role) inner** and loads each checkpoint exactly
once. Filenames follow the standard pattern
`{song}__step_{N}__style_{target}__role_{role}.{wav,mel.pt}` so every
WAV can be traced back to a unique row in `_index.csv`.

**Inputs.** `RUN_SPEC_PATH` (selects checkpoints, songs, styles, roles);
checkpoints under `CHECKPOINT_DIR`.

**Outputs.**

- `versions/<v>/inference_runs/<run_id>/audio/*.wav`
- `versions/<v>/inference_runs/<run_id>/mels/*.mel.pt`
- `versions/<v>/inference_runs/<run_id>/{run_spec.copy.yaml, _summary.csv,
  metrics.json, timing_infos.json}`
- one appended row per WAV in `versions/<v>/inference_runs/_index.csv`

**Action required.** Make sure `run_spec.yaml` lists the candidate steps
you want to compare (typically a 5–10 step window around the best loss).

**Runtime.** Dominated by DDIM sampling; ~1–3 min per `(song × role × step)`
on a T4.

In [ ]:
# run_inference_batch reads EVERYTHING (checkpoint_template, output_root, steps,
# songs, sampling, metrics) from the run_spec — it only takes --run-spec.
# It also computes FAD + (local) F1 best-effort and appends to _index.csv.
!cd {PROJECT_ROOT} && python -m postprocessing.run_inference_batch \
    --run-spec {RUN_SPEC_PATH}


## Cell 5 — FAD (in-Colab, VGGish embeddings)

**What this does.** Computes per-`(target_style, step)` Frechet Audio
Distance between the generated WAVs in this run and the real reference
WAVs configured in the `run_spec.yaml`. Uses pretrained VGGish — runs
fine inside the Colab Python 3.10 image.

**Inputs.** `inference_runs/<run_id>/audio/*.wav`.

**Outputs.** Updates the `fad` column of
`inference_runs/_index.csv` and writes a FAD summary to
`inference_runs/<run_id>/metrics.json`.

**Action required.** None if the previous cell completed.

**Runtime.** ~30 s per 100 WAVs on a T4.

In [ ]:
# Optional standalone FAD (run_inference_batch already wrote FAD into _index.csv).
# fad_eval compares a dir of REAL wavs vs the generated wavs of this run.
# Populate REAL_DIR with the held-out real wavs first (see run_spec comment).
!cd {PROJECT_ROOT} && python -m postprocessing.fad_eval \
    --real "{REAL_DIR}" \
    --generated "{RUNS_ROOT}/{RUN_ID}/audio"


## Cell 6 — Latency (in-Colab, GPU)

**What this does.** Profiles end-to-end inference latency (DDIM + vocoder)
for the steps listed in the run spec.

**Inputs.** Same checkpoints used in Cell 4.

**Outputs.** Per-step `latency_p50` in `_index.csv` and in
`inference_runs/<run_id>/timing_infos.json`.

**Action required.** Make sure the GPU runtime is the same one you intend
to report (latency is hardware-dependent).

**Runtime.** ~2 min.

In [ ]:
# latency_eval is a LIBRARY (evaluate_latency/print_latency_report), not a CLI.
# Per-step timing is written by run_inference_batch to timing_infos.json.
import json
_timing = RUNS_ROOT / RUN_ID / 'timing_infos.json'
if _timing.exists() and _timing.stat().st_size > 2:
    info = json.loads(_timing.read_text())
    from postprocessing.latency_eval import print_latency_report
    for fname, ti in info.items():
        print_latency_report(ti, file_name=fname)
else:
    print('No timing recorded for this run (timing_infos.json empty).')
    print('Latency is hardware-dependent; report it from a dedicated GPU pass if needed.')


## ⚠ Cell 7 — F1 metric requires LOCAL Basic-Pitch — DO NOT RUN IN COLAB

**What this does.** Documents the boundary. Note-level F1 is computed by
transcribing every generated WAV with Basic-Pitch and comparing to the
reference MIDI. **Basic-Pitch needs Python 3.10 + TensorFlow** and is run
only in the local `basic_pitch_env`. Running it here will either fail or
fight with the Colab `tensorflow` build.

**Inputs.** `versions/<v>/inference_runs/<run_id>/audio/*.wav` (already
on your Drive after Cell 4).

**Outputs (filled later, on Windows).** `f1` column of `_index.csv`,
`*.basic_pitch.mid` next to each WAV, and an updated `metrics.json`.

**Action required.** On your local Windows machine, with Drive Desktop
mounted at `G:\My Drive\MusicProject`, run:

```powershell
.\basic_pitch_env\Scripts\Activate.ps1
python -m postprocessing.f1_eval --run-dir "G:\My Drive\MusicProject\versions\Israeli_Shalom_Arik\inference_runs\<run_id>"
```

Then re-sync Drive and continue with Cell 8 here.

**Runtime.** Local; ~10 s per WAV on CPU.

## Cell 8 — Composite-z best-step selection

**What this does.** Ranks every `(step, song, role)` row in
`_index.csv` by `composite_z = z(FAD) + z(1 − F1)` and copies the top-K=3
finalists into `step_selection/step_<N>/{audio,mels,metrics.json}` with a
blind-listening `index.html` and a `ranking.csv`. Falls back to FAD-only
if F1 is missing across the board; ties prefer the earlier step.

**Inputs.** `versions/<v>/inference_runs/_index.csv` (FAD + latency from
Cells 5–6; F1 backfilled by the local step above).

**Outputs.** `versions/<v>/step_selection/{step_<N>/*, ranking.csv, index.html}`.

**Action required.** After this cell, open `step_selection/index.html` in
your browser, play the three finalists blind, and record listening ranks
(1 = best) in `ranking.csv`. The final pick for the deliverables bundle
is the row with the lowest `listening_rank`.

**Runtime.** Seconds.

In [ ]:
!cd {PROJECT_ROOT} && python -m postprocessing.select_best_step \
    --runs-root {RUNS_ROOT} \
    --out-dir {VERSION_DIR} \
    --top-k 3

## Cell 9 — Handoff to local `build_deliverables.py`

**What this does.** Documents the final step, which runs **locally** on
Windows because it consumes the `step_selection/` finalists plus the
(local-only) Basic-Pitch F1 numbers and produces the TA-grading bundle.

**Inputs.** Everything written above plus `deliverables/<run>.yaml`.

**Outputs.** `MusicProject/deliverables/00_overview/index.html` and the
five (now six, including `01b_preprocessing/`) stage folders.

**Action required.** On your local Windows machine, run:

```powershell
.\ml_env\Scripts\Activate.ps1
python build_deliverables.py --config deliverables\Israeli_Shalom_Arik.yaml
```

**Runtime.** Local; ~1 min.